# <font color="#418FDE" size="6.5" uppercase>**Dichte Keras-Modelle**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Erstellen Sequential-Modelle mit Dense-Schichten und passenden Eingabe- und Ausgabeformen. 
- Trainieren kleine Keras-Modelle mit compile, fit, Validierungsdaten und History. 
- Bewerten, regularisieren, speichern und vergleichen Keras-Modelle mit Baselines. 


## **1. Keras Modellaufbau**

### **1.1. Sequential Modelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_01.jpg?v=1787656765" width="250">



>* Schichten verarbeiten Daten Schritt für Schritt
>* Gut geeignet für tabellarische Merkmalsdaten

>* Klare Schichtfolge macht Modelle verständlich
>* Dense-Schichten lernen einfache und komplexe Muster

>* Geeignet für einfache lineare Datenflüsse
>* Klein starten, prüfen und gezielt erweitern



In [ ]:
#@title Python-Code - Sequential Modelle

# Dieses Beispiel baut ein kleines Sequential-Modell.
# Dense-Schichten verarbeiten feste tabellarische Eingabeformen.
# Die Ausgabe zeigt Formen, Training und Genauigkeit.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir verwenden einen kleinen, eingebauten Klassifikationsdatensatz.
iris = load_iris()
features = iris.data.astype("float32")
target = iris.target.astype("int32")

# Diese Prüfung macht die erwartete Eingabeform sichtbar.
if features.shape[1] != 4:
    raise ValueError("Der Datensatz sollte genau vier Merkmale haben.")

# Die Aufteilung hält die Klassenverteilung stabil.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

# Feste Seeds machen das kleine Training besser reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Sequential stapelt die Schichten in klarer Reihenfolge.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(4,)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),
    ]
)

# Compile legt Lernverfahren, Fehlermaß und Metrik fest.
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Fit trainiert das Modell ohne ausführliche Fortschrittsausgabe.
history = model.fit(
    X_train_scaled, y_train, epochs=40, validation_split=0.2, verbose=0
)

# Evaluate prüft das trainierte Modell auf ungesehenen Testdaten.
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

# Kurze Ausgaben verbinden Architektur und Datenformen.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Eingabeform pro Beispiel: {X_train_scaled.shape[1]} Merkmale")
print(f"Ausgabeform pro Beispiel: {model.output_shape[-1]} Klassen")
print(f"Testgenauigkeit: {test_accuracy:.2f}")

# Die Lernkurve zeigt Training und Validierung gemeinsam.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["accuracy"], label="Training")
ax.plot(history.history["val_accuracy"], label="Validierung")
ax.set_title("Lernkurve eines Sequential-Modells")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.legend()
plt.show()



### **1.2. Eingabeformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_02.jpg?v=1787656770" width="250">



>* Eingabeform beschreibt ein einzelnes Datenbeispiel
>* Dense-Schichten erwarten feste Merkmalsvektoren

>* Daten müssen zur Modellform passen
>* Eingabeform ist eine Modellierungsentscheidung

>* Ein Beispiel bestimmt die Eingabeform
>* Datenpipelines müssen Merkmale konsistent liefern



In [ ]:
#@title Python-Code - Eingabeformen verstehen

# Dieses Beispiel zeigt Eingabeformen für Dense-Modelle.
# Wir vergleichen einzelne Beispiele und Stapel.
# Die Ausgabe bestätigt passende Modellformen.

import numpy as np
import tensorflow as tf

# Feste Startwerte machen das Beispiel reproduzierbar.
tf.keras.utils.set_random_seed(42)

# Vier Merkmale beschreiben jeweils eine Kundin.
feature_names = ["Alter", "Einkommen", "Käufe", "Monate"]

# Drei Beispiele bilden einen kleinen Trainingsstapel.
customer_batch = np.array(
    [[32.0, 42000.0, 5.0, 18.0], [45.0, 61000.0, 9.0, 36.0], [28.0, 39000.0, 2.0, 8.0]],
    dtype=np.float32,
)

# Die Eingabeform beschreibt ein einzelnes Beispiel.
input_shape = (customer_batch.shape[1],)

# Diese Prüfung verhindert eine falsche Merkmalsanzahl.
if input_shape != (4,):
    raise ValueError("Erwartet werden genau vier Merkmale pro Beispiel.")

# Ein Dense-Modell erhält die Form ohne Stapeldimension.
model = tf.keras.Sequential(
    [tf.keras.Input(shape=input_shape), tf.keras.layers.Dense(1, activation="sigmoid")]
)

# Eine Vorhersage nutzt den ganzen Stapel gleichzeitig.
predictions = model(customer_batch, training=False).numpy().reshape(-1)

print(f"Merkmale pro Beispiel: {len(feature_names)}")
print(f"Form eines Beispiels: {input_shape}")
print(f"Form des Stapels: {customer_batch.shape}")
print(f"Modell-Eingabeform: {model.input_shape}")
print(f"Modell-Ausgabeform: {model.output_shape}")
print(f"Vorhersagen für {len(predictions)} Beispiele: {np.round(predictions, 3)}")



### **1.3. Passende Ausgabeschichten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_03.jpg?v=1787656767" width="250">



>* Ausgabeschicht richtet sich nach der Aufgabe
>* Vorhersageform muss zu Zielwerten passen

>* Binär: eine Ausgabe als Wahrscheinlichkeit
>* Mehrklassen: eine Ausgabe pro Klasse

>* Mehrere Labels brauchen unabhängige Ausgabeeinheiten
>* Zielvariable bestimmt Aktivierung und Verlustfunktion



In [ ]:
#@title Python-Code - Passende Ausgabeschichten

# Dieses Beispiel vergleicht passende Keras-Ausgabeschichten.
# Jede Aufgabe braucht andere Einheiten und Aktivierungen.
# Die Ausgabeformen zeigen die richtige Modellwahl.

import numpy as np
import tensorflow as tf

# Wir verwenden feste Werte für reproduzierbare Ergebnisse.
tf.keras.utils.set_random_seed(42)

# Drei kleine Eingabebeispiele mit vier Merkmalen.
features = np.array(
    [[0.2, 1.0, -0.4, 0.7], [1.2, -0.3, 0.5, 0.1], [-0.6, 0.4, 1.1, -0.2]],
    dtype=np.float32,
)

# Die Eingabeform muss zur Merkmalsanzahl passen.
if features.shape != (3, 4):
    raise ValueError("Die Beispieldaten müssen drei Zeilen und vier Merkmale haben.")

# Diese Modelle unterscheiden nur die letzte Schicht.
regression_model = tf.keras.Sequential(
    [tf.keras.layers.Input(shape=(4,)), tf.keras.layers.Dense(1)]
)

binary_model = tf.keras.Sequential(
    [tf.keras.layers.Input(shape=(4,)), tf.keras.layers.Dense(1, activation="sigmoid")]
)

multiclass_model = tf.keras.Sequential(
    [tf.keras.layers.Input(shape=(4,)), tf.keras.layers.Dense(3, activation="softmax")]
)

multilabel_model = tf.keras.Sequential(
    [tf.keras.layers.Input(shape=(4,)), tf.keras.layers.Dense(3, activation="sigmoid")]
)

# Wir berechnen Vorhersagen ohne Training.
regression_output = regression_model(features).numpy()
binary_output = binary_model(features).numpy()
multiclass_output = multiclass_model(features).numpy()
multilabel_output = multilabel_model(features).numpy()

# Die Formen zeigen, welche Zielwerte jeweils passen.
print("TensorFlow-Version:", tf.__version__)
print("Regression: Form", regression_output.shape, "Beispiel", round(float(regression_output[0, 0]), 3))
print("Binär: Form", binary_output.shape, "Wert", round(float(binary_output[0, 0]), 3))
print("Mehrklassen: Form", multiclass_output.shape, "Summe", round(float(multiclass_output[0].sum()), 3))
print("Mehrfachlabels: Form", multilabel_output.shape, "Werte", np.round(multilabel_output[0], 3))



## **2. Training mit Keras**

### **2.1. Modell kompilieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_01.jpg?v=1787656754" width="250">



>* Kompilieren legt Lernziel und Strategie fest
>* Es verbindet Modellaufbau mit Training

>* Verlustfunktion misst Fehler und steuert Lernen
>* Sie muss zu Aufgabe und Ausgabe passen

>* Optimierer steuern die Anpassung der Gewichte
>* Metriken machen Trainingsfortschritt sichtbar



In [ ]:
#@title Python-Code - Modell kompilieren

# Dieses Beispiel kompiliert ein kleines Keras-Modell.
# Verlust, Optimierer und Metrik werden sichtbar.
# Die History zeigt den Trainingsverlauf nach compile.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Feste Zufallswerte machen das Beispiel reproduzierbar.
tf.keras.utils.set_random_seed(42)

# Wir erzeugen kleine Regressionsdaten mit einer klaren Regel.
rng = np.random.default_rng(42)
x = rng.uniform(20.0, 100.0, size=(240, 1)).astype("float32")
y = (0.8 * x[:, 0] + 12.0 + rng.normal(0.0, 4.0, 240)).astype("float32")

# Eine einfache Prüfung verhindert unpassende Eingabeformen.
if x.shape != (240, 1) or y.shape != (240,):
    raise ValueError("Die Datenformen passen nicht zum Modell.")

# Trainings- und Validierungsdaten bleiben sauber getrennt.
x_train = x[:180]
y_train = y[:180]
x_val = x[180:]
y_val = y[180:]

# Dense-Schichten erwarten hier genau ein Eingabemerkmal.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(1,)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1),
    ]
)

# Compile legt Lernziel, Lernstrategie und beobachtete Metrik fest.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.03),
    loss="mse",
    metrics=["mae"],
)

# Fit nutzt die Compile-Einstellungen und speichert Werte in History.
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=25,
    verbose=0,
)

# Wir geben nur wenige, wichtige Ergebnisse aus.
print(f"TensorFlow-Version: {tf.__version__}")
print(f"Kompilierte Verlustfunktion: {model.loss}")
print(f"Letzte Trainings-MAE: {history.history['mae'][-1]:.2f}")
print(f"Letzte Validierungs-MAE: {history.history['val_mae'][-1]:.2f}")

# Die Kurven zeigen, was Keras nach compile protokolliert.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["mae"], label="Training")
ax.plot(history.history["val_mae"], label="Validierung")

# Achsen und Legende machen die History leichter lesbar.
ax.set_title("MAE-Verlauf nach dem Kompilieren")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer absoluter Fehler")
ax.legend()
plt.show()



### **2.2. Modelltraining mit fit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_02.jpg?v=1787656758" width="250">



>* fit lässt Modelle aus Trainingsdaten lernen
>* Passende Daten machen Architektur trainierbar

>* Epochen durchlaufen alle Trainingsdaten in Batches
>* Epochenzahl steuert Lernen und Overfitting-Risiko

>* Validierungsdaten prüfen die Verallgemeinerung des Modells
>* History speichert Metriken für spätere Vergleiche



In [ ]:
#@title Python-Code - Modelltraining mit fit

# Dieses Beispiel trainiert ein kleines Keras-Modell.
# fit speichert Lernkurven im History-Objekt.
# Der Plot zeigt Training und Validierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Feste Startwerte machen das Ergebnis gut nachvollziehbar.
np.random.seed(42)
tf.random.set_seed(42)

# Wir nutzen Iris als kleinen Klassifikationsdatensatz.
iris = load_iris()
features = iris.data.astype("float32")
labels = iris.target.astype("int32")

# Eine einfache Prüfung verhindert unklare Formfehler.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Merkmale und Labels haben unterschiedlich viele Zeilen.")

# Die Aufteilung trennt Lernbeispiele von Testbeispielen.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, stratify=labels, random_state=42
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

# Dieses Sequential-Modell hat eine Eingabe und drei Klassen.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(4,)),
        tf.keras.layers.Dense(12, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),
    ]
)

# compile legt Optimierer, Verlustfunktion und Metrik fest.
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# fit trainiert und reserviert Validierungsdaten automatisch.
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=40,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

# evaluate prüft das trainierte Modell auf Testdaten.
test_loss, test_accuracy = model.evaluate(
    X_test_scaled, y_test, verbose=0
)

# Kurze Ausgaben verbinden fit mit messbaren Ergebnissen.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Epochen in history: {len(history.history['loss'])}")
print(f"Testgenauigkeit: {test_accuracy:.2f}")

# Die Lernkurven zeigen Verlust pro Epoche.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["loss"], label="Training")
ax.plot(history.history["val_loss"], label="Validierung")

# Beschriftungen machen den Trainingsverlauf lesbar.
ax.set_title("Verlustkurven aus model.fit")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")
ax.legend()

plt.show()



### **2.3. Trainingsverlauf verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_03.jpg?v=1787656756" width="250">



>* History protokolliert Lernfortschritt pro Epoche
>* Loss und Metriken zeigen Modellleistung

>* Training und Validierung gemeinsam beurteilen
>* Abweichungen zeigen Über- oder Unteranpassung

>* Verlaufsmuster statt Einzelwerte bewerten
>* Maßnahmen für bessere Generalisierung ableiten



In [ ]:
#@title Python-Code - Trainingsverlauf verstehen

# Wir untersuchen den Trainingsverlauf eines Keras-Modells.
# History speichert Verlust und Genauigkeit pro Epoche.
# Die Kurven zeigen Lernen und mögliche Überanpassung.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Feste Startwerte machen das Beispiel reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Ein kleines Klassifikationsproblem bleibt übersichtlich.
data = load_breast_cancer()
features = data.data.astype("float32")
target = data.target.astype("float32")

# Diese Prüfung verhindert unklare Formfehler.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung trennt Training und Validierung sauber.
X_train, X_val, y_train, y_val = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")

# Ein kleines Dense-Modell reicht für den Verlauf.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

# Compile legt Optimierer, Verlust und Metrik fest.
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# Fit liefert ein History-Objekt mit Messwerten.
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=25,
    batch_size=32,
    verbose=0,
)

# Die gespeicherten Schlüssel zeigen verfügbare Kurven.
history_keys = sorted(history.history.keys())
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"History-Schlüssel: {history_keys}")

# Anfang und Ende machen den Lernfortschritt greifbar.
start_loss = history.history["loss"][0]
end_loss = history.history["loss"][-1]
end_val_accuracy = history.history["val_accuracy"][-1]
print(f"Trainingsverlust: {start_loss:.3f} -> {end_loss:.3f}")
print(f"Validierungsgenauigkeit am Ende: {end_val_accuracy:.3f}")

# Eine einzelne Achse vergleicht Training und Validierung.
epochs = np.arange(1, len(history.history["loss"]) + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history.history["loss"], label="Training")
ax.plot(epochs, history.history["val_loss"], label="Validierung")

# Beschriftungen helfen beim Lesen der Lernkurven.
ax.set_title("Trainingsverlauf: Verlust pro Epoche")
ax.set_xlabel("Epoche")
ax.set_ylabel("Binary-Crossentropy-Verlust")
ax.legend()
plt.show()



## **3. Bewertung und Vergleich**

### **3.1. Regularisierung gegen Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_01.jpg?v=1787656760" width="250">



>* Overfitting lernt Trainingsdaten statt Muster
>* Regularisierung verbessert Vorhersagen auf neuen Daten

>* Dropout und Gewichtsregeln verringern Overfitting
>* Kleinere Modelle und Early Stopping schützen

>* Regularisierung immer mit Baseline und Validierung prüfen
>* Varianten vergleichen, um Underfitting zu vermeiden



In [ ]:
#@title Python-Code - Regularisierung gegen Overfitting

# Dieses Beispiel vergleicht Regularisierung gegen Overfitting.
# Dropout und L2 sollen Validierungsleistung stabilisieren.
# Die Kurven zeigen Training und Validierung.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_classification

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import sklearn

# Wir erzeugen kleine Klassifikationsdaten mit etwas Rauschen.
features, labels = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=6,
    n_redundant=4,
    flip_y=0.12,
    class_sep=0.8,
    random_state=42,
)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (1200, 20):
    raise ValueError("Die erzeugten Daten haben eine unerwartete Form.")

# Die Aufteilung hält beide Klassen ähnlich verteilt.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TensorFlow erhält feste Zufallswerte für reproduzierbares Training.
tf.keras.utils.set_random_seed(42)

# Ein bewusst großes Modell kann leichter overfitten.
plain_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(20,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

# Das regularisierte Modell nutzt L2 und Dropout.
regularized_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(20,)),
        tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

# Beide Modelle verwenden dieselbe einfache Lernaufgabe.
plain_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

regularized_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Wir trainieren kurz und speichern die History.
plain_history = plain_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.25,
    epochs=35,
    batch_size=32,
    verbose=0,
)

regularized_history = regularized_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.25,
    epochs=35,
    batch_size=32,
    verbose=0,
)

# Eine einfache Baseline sagt immer die häufigste Klasse voraus.
baseline_class = int(np.bincount(y_train).argmax())
baseline_predictions = np.full_like(y_test, baseline_class)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

# Testwerte vergleichen Baseline, unregularisiert und regularisiert.
plain_test_accuracy = plain_model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0,
)[1]

regularized_test_accuracy = regularized_model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0,
)[1]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Baseline-Testgenauigkeit: {baseline_accuracy:.3f}")
print(f"Ohne Regularisierung: {plain_test_accuracy:.3f}")
print(f"Mit Dropout und L2: {regularized_test_accuracy:.3f}")

# Die Validierungskurven zeigen die Generalisierung über Epochen.
epochs = np.arange(1, 36)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, plain_history.history["val_accuracy"], label="ohne Regularisierung")
ax.plot(epochs, regularized_history.history["val_accuracy"], label="mit Dropout und L2")

ax.set_title("Validierungsgenauigkeit mit und ohne Regularisierung")
ax.set_xlabel("Epoche")
ax.set_ylabel("Validierungsgenauigkeit")
ax.legend()
plt.show()



### **3.2. Vorhersagen interpretieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_02.jpg?v=1787656761" width="250">



>* Regression: Plausibilität und systematische Fehler prüfen
>* Klassifikation: Wahrscheinlichkeiten im Kontext bewerten

>* Ausgabeverteilung zeigt Sicherheit der Klassifikation
>* Unsichere Grenzfälle gezielt weiter prüfen

>* Vorhersagen mit Baselines und Fachwissen prüfen
>* Fehler und Grenzfälle zeigen Praxistauglichkeit



In [ ]:
#@title Python-Code - Vorhersagen interpretieren

# Wir interpretieren Vorhersagen eines kleinen Keras-Modells.
# Wahrscheinlichkeiten zeigen Sicherheit statt nur Klassen.
# Die Grafik markiert sichere und unsichere Fälle.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Feste Startwerte machen Training und Auswahl nachvollziehbar.
np.random.seed(42)
tf.random.set_seed(42)

# Ein kleines Klassifikationsproblem bleibt übersichtlich.
data = load_breast_cancer()
features = data.data
target = data.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung nutzt Stratifikation für faire Klassenanteile.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Ein dichtes Modell liefert eine Wahrscheinlichkeit pro Beispiel.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(12, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

# Binary Crossentropy passt zu zwei Klassen.
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# Validierungsdaten helfen, Trainingsergebnisse einzuordnen.
history = model.fit(
    X_train_scaled, y_train, epochs=25, batch_size=32, verbose=0,
    validation_split=0.2
)

# Wahrscheinlichkeiten werden in Klassen und Sicherheit übersetzt.
probabilities = model.predict(X_test_scaled, verbose=0).ravel()
predicted_classes = (probabilities >= 0.5).astype(int)
confidence = np.maximum(probabilities, 1 - probabilities)

# Eine einfache Baseline sagt immer die häufigste Trainingsklasse.
baseline_class = int(np.bincount(y_train).argmax())
baseline_predictions = np.full_like(y_test, baseline_class)

model_accuracy = accuracy_score(y_test, predicted_classes)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
uncertain_count = int(np.sum(confidence < 0.65))

# Drei Kennzahlen verbinden Vorhersagen mit Modellvergleich.
print(f"scikit-learn-Version: 1.9.0")
print(f"Modellgenauigkeit: {model_accuracy:.3f}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.3f}")
print(f"Unsichere Testfälle unter 65 Prozent Sicherheit: {uncertain_count}")

# Die unsichersten Beispiele zeigen Grenzfälle der Entscheidung.
order = np.argsort(confidence)[:8]
shown_confidence = confidence[order]
shown_correct = predicted_classes[order] == y_test[order]

colors = np.where(shown_correct, "tab:green", "tab:red")
labels = [f"Fall {i + 1}" for i in range(len(order))]

# Ein Balkendiagramm macht Sicherheit und Fehler sichtbar.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, shown_confidence, color=colors)
ax.axhline(0.65, color="black", linestyle="--", label="Unsicherheitsgrenze")

ax.set_title("Unsicherste Modellvorhersagen im Testset")
ax.set_xlabel("Ausgewählte Testfälle")
ax.set_ylabel("Vorhersagesicherheit")
ax.set_ylim(0.45, 1.0)
ax.legend()

plt.show()



### **3.3. Keras Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_03.jpg?v=1787656763" width="250">



>* Mini Projekt: Daten, Baseline, Modell, Bewertung
>* Mehrwert entsteht erst gegenüber einfachen Vergleichen

>* Generalisierung mit Validierungs- und Testdaten prüfen
>* Vorhersagefehler im Anwendungskontext interpretieren

>* Modell speichern und nachvollziehbar dokumentieren
>* Varianten verantwortungsvoll mit Baselines vergleichen



In [ ]:
#@title Python-Code - Keras Mini Projekt

# Dieses Mini Projekt vergleicht Baseline und Keras-Modell.
# Regularisierung hilft beim fairen Modellvergleich.
# Am Ende sehen wir Testwerte und Lernkurven.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Feste Zufallswerte machen das Ergebnis reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
features = data.data
target = data.target

# Eine einfache Prüfung verhindert stille Formfehler.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Zuerst trennen wir einen unabhängigen Testteil ab.
features_train_full, features_test, target_train_full, target_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=42
)

# Danach entsteht ein Validierungsteil für den Modellvergleich.
features_train, features_val, target_train, target_val = train_test_split(
    features_train_full, target_train_full, test_size=0.25, stratify=target_train_full,
    random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_val_scaled = scaler.transform(features_val)
features_test_scaled = scaler.transform(features_test)

# Die Baseline sagt immer die häufigste Trainingsklasse voraus.
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(features_train_scaled, target_train)
baseline_predictions = baseline.predict(features_test_scaled)

# Ein kleines Dense-Modell bleibt übersichtlich.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(features_train_scaled.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Compile legt Optimierer, Verlustfunktion und Kennzahl fest.
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# Fit trainiert einmal und speichert die Lernkurven.
history = model.fit(
    features_train_scaled, target_train, validation_data=(features_val_scaled, target_val),
    epochs=35, batch_size=16, verbose=0
)

# Die Testdaten bleiben bis zur finalen Bewertung unberührt.
test_loss, test_accuracy = model.evaluate(
    features_test_scaled, target_test, verbose=0
)

# Wir vergleichen nur wenige, klare Kennzahlen.
baseline_accuracy = accuracy_score(target_test, baseline_predictions)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline-Testgenauigkeit: {baseline_accuracy:.3f}")
print(f"Keras-Testgenauigkeit: {test_accuracy:.3f}")

# Die Lernkurven zeigen Training und Validierung gemeinsam.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["accuracy"], label="Training")
ax.plot(history.history["val_accuracy"], label="Validierung")

# Achsen und Legende machen den Vergleich lesbar.
ax.set_title("Keras Mini Projekt: Genauigkeit pro Epoche")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Dichte Keras-Modelle**</font>


In this lecture, you learned to:
- Erstellen Sequential-Modelle mit Dense-Schichten und passenden Eingabe- und Ausgabeformen. 
- Trainieren kleine Keras-Modelle mit compile, fit, Validierungsdaten und History. 
- Bewerten, regularisieren, speichern und vergleichen Keras-Modelle mit Baselines. 

In the next Module (Module 16), we will go over 'CNNs mit Keras'